In [1]:
import tensorflow as tf
import numpy as np
import pickle
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [2]:
# Direct Tamil Bi-GRU
bi_gru_model = tf.keras.models.load_model(
    "bi_gru_1111.keras", compile=False
)

# English LSTM & BiLSTM
lstm_model = tf.keras.models.load_model(
    "baseline_lstm_glove.keras", compile=False
)

bilstm_model = tf.keras.models.load_model(
    "bilstm_glove_finetuned.keras", compile=False
)


In [3]:
# Tamil vocab
with open("word2idxlarge.pkl", "rb") as f:
    ta_word2idx = pickle.load(f)

with open("idx2wordlarge.pkl", "rb") as f:
    ta_idx2word = pickle.load(f)

# English tokenizers
with open("tokenizer_baseline.pkl", "rb") as f:
    en_tokenizer_lstm = pickle.load(f)

with open("tokenizer_bilstm_glove.pkl", "rb") as f:
    en_tokenizer_bilstm = pickle.load(f)

TA_VOCAB_SIZE = len(ta_word2idx)
EN_VOCAB_SIZE = len(en_tokenizer_lstm.word_index) + 1


In [11]:
import os
from transformers import MarianMTModel, MarianTokenizer

# 1. Define folder names for local storage
ta_en_path = "./saved_nmt/tamil_to_english"
en_ta_path = "./saved_nmt/english_to_tamil"

# 2. Create the folders if they don't exist
os.makedirs(ta_en_path, exist_ok=True)
os.makedirs(en_ta_path, exist_ok=True)

print("Downloading and saving models... please wait.")

# 3. Download and save Tamil -> English
tokenizer_ta_en = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-dra-en")
model_ta_en = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-dra-en")
tokenizer_ta_en.save_pretrained(ta_en_path)
model_ta_en.save_pretrained(ta_en_path)

# 4. Download and save English -> Tamil
tokenizer_en_ta = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-dra")
model_en_ta = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-dra")
tokenizer_en_ta.save_pretrained(en_ta_path)
model_en_ta.save_pretrained(en_ta_path)

print(f"✅ Success! Models saved to: {os.path.abspath('./saved_nmt')}")

C:\Users\Abisheak\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\Abisheak\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ Success! Models saved to: E:\CCP2_TAMtoENG_AAC\tamil_next_word_app\new ui\saved_nmt


In [13]:
import torch
from transformers import MarianMTModel, MarianTokenizer

# Explicitly use CPU
device = "cpu"

# Paths to your saved models
ta_en_dir = "./saved_nmt/tamil_to_english"
en_ta_dir = "./saved_nmt/english_to_tamil"

print("Loading models from local disk (Offline Mode)...")

# Load from LOCAL folders
tokenizer_ta_en = MarianTokenizer.from_pretrained(ta_en_dir, local_files_only=True)
model_ta_en = MarianMTModel.from_pretrained(ta_en_dir, local_files_only=True).to(device)

tokenizer_en_ta = MarianTokenizer.from_pretrained(en_ta_dir, local_files_only=True)
model_en_ta = MarianMTModel.from_pretrained(en_ta_dir, local_files_only=True).to(device)

def translate_offline(text, direction="ta_en"):
    if direction == "ta_en":
        inputs = tokenizer_ta_en(text, return_tensors="pt", padding=True)
        with torch.no_grad():
            output = model_ta_en.generate(**inputs)
        return tokenizer_ta_en.batch_decode(output, skip_special_tokens=True)[0]
    else:
        # English to Tamil requires the >>tam<< prefix
        inputs = tokenizer_en_ta(">>tam<< " + text, return_tensors="pt", padding=True)
        with torch.no_grad():
            output = model_en_ta.generate(**inputs)
        return tokenizer_en_ta.batch_decode(output, skip_special_tokens=True)[0]

# Presentation Test
print(f"Tamil to English: {translate_offline('மிக்க நன்றி.', 'ta_en')}")
print(f"English to Tamil: {translate_offline('following', 'en_ta')}")

Loading models from local disk (Offline Mode)...
Tamil to English: Thank you very much.
English to Tamil: தொடரவும்


In [5]:
def translate(text, tokenizer, model):
    inputs = tokenizer(text, return_tensors="pt", padding=True)
    outputs = model.generate(**inputs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [6]:
MAX_SEQ_LEN_TA = 12
UNK = "<UNK>"

def predict_tamil_bigru(text, top_k=3):
    tokens = text.strip().split()
    encoded = [ta_word2idx.get(w, ta_word2idx.get(UNK, 0)) for w in tokens]
    padded = pad_sequences([encoded], maxlen=MAX_SEQ_LEN_TA, padding="pre")

    probs = bi_gru_model.predict(padded, verbose=0)[0]
    top_idx = np.argsort(probs)[-top_k:][::-1]

    results = [(ta_idx2word[i], float(probs[i])) for i in top_idx]
    top1_prob = float(probs[top_idx[0]])

    return results, top1_prob


In [7]:
MAX_SEQ_LEN_EN = 15

def predict_english(model, tokenizer, text, top_k=3):
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=MAX_SEQ_LEN_EN, padding="pre")

    probs = model.predict(padded, verbose=0)[0]
    top_idx = np.argsort(probs)[-top_k:][::-1]

    inv_vocab = {v: k for k, v in tokenizer.word_index.items()}
    results = [(inv_vocab.get(i, ""), float(probs[i])) for i in top_idx]
    top1_prob = float(probs[top_idx[0]])

    return results, top1_prob


In [14]:
def translation_pipeline_predict(text_ta, model, tokenizer, top_k=3):
    # Tamil → English (offline)
    text_en = translate_offline(text_ta, direction="ta_en")

    # Predict next English words
    en_preds, top1_prob = predict_english(
        model, tokenizer, text_en, top_k
    )

    # English → Tamil (offline)
    ta_preds = []
    for word, prob in en_preds:
        ta_word = translate_offline(word, direction="en_ta")
        ta_preds.append((ta_word, prob))

    return ta_preds, top1_prob


In [15]:
def predict_all_models(current_sentence):
    results = {}

    # Direct Tamil Bi-GRU
    results["Direct Tamil Bi-GRU"] = predict_tamil_bigru(
        current_sentence
    )

    # LSTM (translation-based)
    results["LSTM (Translate-based)"] = translation_pipeline_predict(
        current_sentence,
        lstm_model,
        en_tokenizer_lstm
    )

    # BiLSTM (translation-based)
    results["BiLSTM (Translate-based)"] = translation_pipeline_predict(
        current_sentence,
        bilstm_model,
        en_tokenizer_bilstm
    )

    return results


In [16]:
sentence = "நான் இன்று"

for step in range(3):
    print(f"\nSTEP {step + 1}")
    print("Current sentence:", sentence)

    outputs = predict_all_models(sentence)

    for model_name, (preds, conf) in outputs.items():
        print(f"\n{model_name}")
        print("Top-3 predictions:", preds)
        print("Top-1 probability:", round(conf, 3))

    # Simulate button click → choose first Bi-GRU suggestion
    sentence += " " + outputs["Direct Tamil Bi-GRU"][0][0][0]



STEP 1
Current sentence: நான் இன்று

Direct Tamil Bi-GRU
Top-3 predictions: [('நடந்தான்', 0.268692284822464), ('தமிழ்', 0.01754453405737877), ('<UNK>', 0.01690860465168953)]
Top-1 probability: 0.269

LSTM (Translate-based)
Top-3 predictions: [('சமீபத்திய', 0.41313302516937256), ('இதற்கு', 0.1370944231748581), ('இற்றைப்படுத்தல்', 0.12285315245389938)]
Top-1 probability: 0.413

BiLSTM (Translate-based)
Top-3 predictions: [('ஆலோசனைகள்', 0.4425280690193176), ('இதற்கு', 0.12497065961360931), ('தேவை', 0.06954677402973175)]
Top-1 probability: 0.443

STEP 2
Current sentence: நான் இன்று நடந்தான்

Direct Tamil Bi-GRU
Top-3 predictions: [('கேட்டேன்', 0.06813710182905197), ('என்று', 0.06391183286905289), ('பார்த்தேன்', 0.04516308009624481)]
Top-1 probability: 0.068

LSTM (Translate-based)
Top-3 predictions: [('இதற்கு', 0.33838459849357605), ('சமீபத்திய', 0.22377896308898926), ('போல', 0.06394091248512268)]
Top-1 probability: 0.338

BiLSTM (Translate-based)
Top-3 predictions: [('இதற்கு', 0.54594981